# ad spend vs signups — poking at it

(scratch, don't trust anything below cell 40)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
import seaborn as sns
sns.set()

In [3]:
pd.set_option('display.max_rows', 200)

In [4]:
df = pd.read_csv('data/spend.csv')

In [ ]:
df = pd.read_csv('../data/spend.csv')

In [ ]:
import os
os.listdir('..')

In [ ]:
os.listdir('../data')

In [5]:
df = pd.read_csv('../data/spend_2025_q1_FINAL.csv')

In [6]:
df.head()

In [7]:
df.shape

In [8]:
df.columns

In [9]:
df.dtypes

In [10]:
df.head(30)

In [11]:
signups = pd.read_csv('../data/signups.csv')

In [12]:
signups.head()

In [13]:
signups.shape

In [14]:
df.channel.unique()

In [15]:
signups.source.unique()

In [16]:
# channel vs source — not the same vocabulary, of course
set(df.channel.unique()) - set(signups.source.unique())

In [17]:
df['date'] = pd.to_datetime(df['date'])

In [18]:
signups['date'] = pd.to_datetime(signups['created_at']).dt.floor('D')

In [19]:
df.date.min(), df.date.max()

In [20]:
signups.date.min(), signups.date.max()

In [ ]:
df = df[df.date >= '2025-01-01']

In [21]:
df.shape

In [22]:
df.spend.describe()

In [23]:
df[df.spend < 0]

In [24]:
df[df.spend < 0].shape

In [25]:
df = df[df.spend >= 0]

In [26]:
df.spend.describe()

In [27]:
df.spend.hist(bins=50)

In [28]:
np.log1p(df.spend).hist(bins=50)

In [29]:
df.channel.value_counts()

In [30]:
df.channel = df.channel.str.lower().str.strip()

In [31]:
df.channel.value_counts()

In [32]:
df.channel = df.channel.replace({'fb': 'facebook', 'facebook ads': 'facebook', 'goog': 'google', 'google ads': 'google'})

In [33]:
df.channel.value_counts()

In [34]:
signups.source = signups.source.str.lower().str.strip()

In [35]:
signups.source.value_counts()

In [36]:
signups.source = signups.source.replace({'fb': 'facebook', 'organic search': 'organic', 'seo': 'organic'})

In [25]:
signups.source.value_counts()

In [38]:
set(df.channel.unique()) - set(signups.source.unique())

In [39]:
set(signups.source.unique()) - set(df.channel.unique())

In [ ]:
# 'referral' has no spend, 'affiliate' has no signups. leaving both in for now

In [ ]:
df.duplicated(['date', 'channel', 'campaign']).sum()

In [ ]:
df[df.duplicated(['date','channel','campaign'], keep=False)].sort_values(['date','channel']).head(20)

In [40]:
df = df.groupby(['date','channel','campaign'], as_index=False).spend.sum()

In [41]:
df.shape

In [42]:
!ls -la ../data | head

In [43]:
daily = df.groupby(['date','channel'], as_index=False).spend.sum()

In [44]:
daily.head()

In [45]:
s = signups.groupby(['date','source'], as_index=False).size().rename(columns={'source':'channel','size':'signups'})

In [46]:
s.head()

In [47]:
m = daily.merge(s, on=['date','channel'], how='outer')

In [ ]:
m.shape

In [48]:
m.isna().sum()

In [49]:
m = m.fillna(0)

In [50]:
m.head()

In [51]:
m.groupby('date').spend.sum().plot()

In [52]:
m.groupby('date').spend.sum().plot(figsize=(14,4))

In [53]:
m.groupby('date').signups.sum().plot(figsize=(14,4))

In [54]:
fig, ax = plt.subplots(figsize=(14,4))
m.groupby('date').spend.sum().plot(ax=ax)
m.groupby('date').signups.sum().plot(ax=ax, secondary_y=True)

In [35]:
fig, ax = plt.subplots(figsize=(14,4))
m.groupby('date').spend.sum().rolling(7).mean().plot(ax=ax)
m.groupby('date').signups.sum().rolling(7).mean().plot(ax=ax, secondary_y=True)
plt.title('7d rolling')

In [56]:
for ch, g in m.groupby('channel'):
    g.set_index('date').signups.rolling(7).mean().plot(figsize=(14,4), label=ch)
plt.legend()

In [57]:
m[m.channel=='facebook'].set_index('date')[['spend','signups']].rolling(7).mean().plot(figsize=(14,4), secondary_y='signups')

In [58]:
m[m.channel=='google'].set_index('date')[['spend','signups']].rolling(7).mean().plot(figsize=(14,4), secondary_y='signups')

In [59]:
m[m.channel=='organic'].set_index('date')[['spend','signups']].rolling(7).mean().plot(figsize=(14,4), secondary_y='signups')

TODO: pull the plotting out into a function, this is getting silly

In [60]:
def two_axis(ch, win=7):
    sub = m[m.channel==ch].set_index('date')[['spend','signups']].rolling(win).mean()
    ax = sub.spend.plot(figsize=(14,4))
    sub.signups.plot(ax=ax, secondary_y=True)
    ax.set_title(ch)
    return ax

In [61]:
two_axis('facebook')

In [ ]:
two_axis('google', 14)

In [ ]:
m['cpa'] = m.spend / m.signups.replace(0, np.nan)

In [62]:
m.cpa.describe()

In [63]:
m.groupby('channel').cpa.median()

In [64]:
m[m.cpa > 500].shape

In [65]:
m[m.cpa > 500].head(20)

In [66]:
m.loc[m.cpa > 500, 'cpa'] = np.nan

In [67]:
m.groupby('channel').cpa.median()

In [68]:
m['dow'] = m.date.dt.dayofweek

In [69]:
m['week'] = m.date.dt.isocalendar().week.astype(int)

In [70]:
m['month'] = m.date.dt.month

In [71]:
m['is_weekend'] = m.dow.isin([5,6]).astype(int)

In [72]:
m.groupby('dow').signups.mean()

In [67]:
m.groupby('is_weekend').cpa.median()

In [74]:
for lag in [1,2,3,7,14]:
    m[f'spend_lag{lag}'] = m.groupby('channel').spend.shift(lag)

In [75]:
m[[c for c in m.columns if 'lag' in c]].isna().mean()

In [76]:
m['spend_7d'] = m.groupby('channel').spend.transform(lambda s: s.rolling(7, min_periods=1).sum())

In [77]:
m['spend_28d'] = m.groupby('channel').spend.transform(lambda s: s.rolling(28, min_periods=1).sum())

In [78]:
m['saturation'] = m.spend_7d / m.spend_28d.replace(0, np.nan)

In [79]:
m.saturation.describe()

In [80]:
# wait — the outer merge put zero-spend rows in for organic, that's poisoning the lags
m[m.channel=='organic'].spend.sum()

In [81]:
m = m[~((m.channel=='organic') & (m.spend==0))]

In [82]:
m.shape

In [83]:
for lag in [1,2,3,7,14]:
    m[f'spend_lag{lag}'] = m.groupby('channel').spend.shift(lag)
m['spend_7d'] = m.groupby('channel').spend.transform(lambda s: s.rolling(7, min_periods=1).sum())
m['spend_28d'] = m.groupby('channel').spend.transform(lambda s: s.rolling(28, min_periods=1).sum())
m['saturation'] = m.spend_7d / m.spend_28d.replace(0, np.nan)

In [84]:
m.saturation.describe()

In [ ]:
model_df = m.dropna().copy()

In [ ]:
model_df.shape

In [ ]:
model_df.head()

In [85]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, r2_score

In [86]:
feats = [c for c in model_df.columns if c not in ('date','channel','signups','cpa')]
feats

In [56]:
X = pd.get_dummies(model_df[feats + ['channel']], columns=['channel'])
y = model_df.signups

In [88]:
tscv = TimeSeriesSplit(n_splits=5)

In [89]:
scores = []
for tr, te in tscv.split(X):
    mdl = Ridge(alpha=1.0).fit(X.iloc[tr], y.iloc[tr])
    scores.append(r2_score(y.iloc[te], mdl.predict(X.iloc[te])))
scores

In [90]:
np.mean(scores)

In [91]:
# hang on, spend_7d includes today. that's leakage against same-day signups.
model_df[['date','channel','spend','spend_7d']].head(10)

In [92]:
m['spend_7d'] = m.groupby('channel').spend.transform(lambda s: s.shift(1).rolling(7, min_periods=1).sum())

In [93]:
m['spend_28d'] = m.groupby('channel').spend.transform(lambda s: s.shift(1).rolling(28, min_periods=1).sum())

In [94]:
m['saturation'] = m.spend_7d / m.spend_28d.replace(0, np.nan)

In [95]:
model_df = m.dropna().copy()
X = pd.get_dummies(model_df[feats + ['channel']], columns=['channel'])
y = model_df.signups
model_df.shape

In [96]:
scores = []
for tr, te in tscv.split(X):
    mdl = Ridge(alpha=1.0).fit(X.iloc[tr], y.iloc[tr])
    scores.append(r2_score(y.iloc[te], mdl.predict(X.iloc[te])))
np.mean(scores)

In [97]:
scores = []
for tr, te in tscv.split(X):
    mdl = Ridge(alpha=10.0).fit(X.iloc[tr], y.iloc[tr])
    scores.append(r2_score(y.iloc[te], mdl.predict(X.iloc[te])))
np.mean(scores)

In [98]:
scores = []
for tr, te in tscv.split(X):
    mdl = Ridge(alpha=100.0).fit(X.iloc[tr], y.iloc[tr])
    scores.append(r2_score(y.iloc[te], mdl.predict(X.iloc[te])))
np.mean(scores)

In [99]:
scores = []
for tr, te in tscv.split(X):
    mdl = Ridge(alpha=300.0).fit(X.iloc[tr], y.iloc[tr])
    scores.append(r2_score(y.iloc[te], mdl.predict(X.iloc[te])))
np.mean(scores)

In [100]:
ALPHA = 100.0

In [101]:
ridge = Ridge(alpha=ALPHA).fit(X, y)

In [102]:
pd.Series(ridge.coef_, index=X.columns).sort_values().head(12)

In [103]:
pd.Series(ridge.coef_, index=X.columns).sort_values().tail(12)

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

In [104]:
gb_scores = []
for tr, te in tscv.split(X):
    g = GradientBoostingRegressor(random_state=0).fit(X.iloc[tr], y.iloc[tr])
    gb_scores.append(r2_score(y.iloc[te], g.predict(X.iloc[te])))
np.mean(gb_scores)

In [105]:
gb = GradientBoostingRegressor(random_state=0).fit(X, y)

In [66]:
pd.Series(gb.feature_importances_, index=X.columns).sort_values(ascending=False).head(15)

In [107]:
pred = gb.predict(X)

In [108]:
mean_absolute_error(y, pred)

In [109]:
plt.figure(figsize=(14,4))
plt.plot(model_df.date.values, y.values, label='actual')
plt.plot(model_df.date.values, pred, label='fit')
plt.legend()

In [110]:
other = pd.read_excel('../data/from_marketing_dont_delete.xlsx')

In [ ]:
other.head()

In [111]:
other.columns.tolist()

In [ ]:
model_df['pred'] = pred

In [ ]:
model_df['resid'] = model_df.signups - model_df.pred

In [112]:
model_df.resid.describe()

In [113]:
model_df.groupby('channel').resid.mean()

In [114]:
model_df.groupby('channel').apply(lambda g: mean_absolute_error(g.signups, g.pred))

In [115]:
model_df.set_index('date').resid.plot(figsize=(14,4))

In [116]:
model_df.groupby('month').resid.mean()

In [117]:
model_df.groupby('dow').resid.mean()

In [118]:
sns.scatterplot(data=model_df, x='pred', y='signups', hue='channel')

In [119]:
worst = model_df.reindex(model_df.resid.abs().sort_values(ascending=False).index).head(20)
worst[['date','channel','spend','signups','pred','resid']]

In [120]:
worst.channel.value_counts()

In [121]:
worst.date.dt.month.value_counts()

In [122]:
ridge_pred = ridge.predict(X)
mean_absolute_error(y, ridge_pred)

In [123]:
plt.figure(figsize=(14,4))
plt.plot(model_df.date.values, y.values, label='actual')
plt.plot(model_df.date.values, ridge_pred, label='ridge')
plt.plot(model_df.date.values, pred, label='gb')
plt.legend()

In [124]:
pd.DataFrame({'ridge': [mean_absolute_error(y, ridge_pred)], 'gb': [mean_absolute_error(y, pred)]})

## notes

gb wins on MAE but the residuals are seasonal, so it's probably just memorising Q1. don't ship.

In [125]:
model_df.to_csv('../data/model_df.csv', index=False)

In [126]:
import joblib
joblib.dump(gb, '../models/gb.pkl')

In [127]:
# scratch below

In [128]:
m.channel.value_counts()

In [129]:
len(feats)

In [ ]:
X.shape

In [ ]:
# X.to_parquet('../data/X.parquet')

In [ ]:
model_df.date.max()

In [ ]:
del other

In [ ]:
print('ok')